# Evidence Retrieval Experiment Framework

This notebook acts as the central experiment framework for the dissertation.

It does not perform exploratory analysis of AVeriTeC; this is covered in the separate data inspection notebook. Instead, this notebook:

1. defines the experimental and reproducibility settings;
2. loads the prepared claims and candidate evidence collection;
3. executes each retrieval configuration through a common interface;
4. evaluates the resulting rankings using the same metrics;
5. records the configuration and environment associated with each run;

Retrieval implementations are contained in separate Python modules so that the experimental procedure remains consistent across methods.

## 1. Setup and Imports

### 1.1. Imports

In [15]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
from datetime import datetime, timezone

import os
import sys
import json
import time
import random
import psutil
import threading
import subprocess
from time import perf_counter
import matplotlib.pyplot as plt

import gc
import nltk
import torch
import numpy as np
import pandas as pd
import transformers
import sentence_transformers
from tqdm.auto import tqdm
from pynvml import nvmlInit, nvmlDeviceGetHandleByIndex, nvmlDeviceGetProcessUtilization, NVMLError


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### 1.2. Resource Monitor Utils

In [16]:
class ResourceMonitor:
    def __init__(self, interval=1.0):
        nvmlInit()
        self.gpu_handle = nvmlDeviceGetHandleByIndex(0)
        self.pid = os.getpid()
        self.last_gpu_timestamp = 0
        self.interval = interval
        self.process = psutil.Process(os.getpid())
        self.samples = []
        self.running = False
        self.thread = None

    def _sample(self):
        self.process.cpu_percent(None)

        start = time.perf_counter()



        while self.running:
            elapsed = time.perf_counter() - start
            cpu = (self.process.cpu_percent(None) / psutil.cpu_count())
            ram_gb = (self.process.memory_info().rss / 1024**3)
            gpu_gb = None
            gpu_util = self._gpu_process_utilisation() if torch.cuda.is_available() else None

            if torch.cuda.is_available():
                gpu_gb = (torch.cuda.memory_allocated() / 1024**3)

            self.samples.append({"time": elapsed, "cpu": cpu, "ram_gb": ram_gb, "gpu_util": gpu_util, "gpu_gb": gpu_gb})
            time.sleep(self.interval)

    def start(self):
        self.samples = []
        self.running = True
        self.thread = threading.Thread(target=self._sample, daemon=True)
        self.thread.start()

    def stop(self):
        self.running = False

        if self.thread is not None:
            self.thread.join()

        return pd.DataFrame(self.samples)

    def _gpu_process_utilisation(self):
        try:
            samples = nvmlDeviceGetProcessUtilization(self.gpu_handle, self.last_gpu_timestamp)

            gpu_util = 0.0

            for sample in samples:
                if sample.pid == self.pid:
                    gpu_util = float(sample.smUtil)

                    self.last_gpu_timestamp = max(self.last_gpu_timestamp, sample.timeStamp)

            return gpu_util

        except NVMLError:
            return 0.0


def plot_resources(data, title, smoothing_window=30):
    plot_data = data.copy()

    for column in ["cpu", "ram_gb", "gpu_gb", "gpu_util"]:
        if column in plot_data.columns:
            plot_data[column] = plot_data[column].rolling(window=smoothing_window, min_periods=1, center=True).mean()

    fig, ax = plt.subplots()

    ax.plot(plot_data["time"], plot_data["cpu"], label="CPU utilisation (%)")
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("CPU utilisation (%)")
    ax.set_title(title)
    ax.legend()

    plt.show()

    fig, ax = plt.subplots()

    ax.plot(plot_data["time"], plot_data["ram_gb"], label="RAM (GB)")

    if plot_data["gpu_gb"].notna().any():
        ax.plot(plot_data["time"], plot_data["gpu_gb"], label="GPU VRAM (GB)")

    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Memory (GB)")
    ax.set_title(title)
    ax.legend()

    plt.show()

    if "gpu_util" in plot_data.columns and plot_data["gpu_util"].notna().any():
        fig, ax = plt.subplots()

        ax.plot(plot_data["time"], plot_data["gpu_util"], label="GPU utilisation (%)")

        ax.set_xlabel("Time (s)")
        ax.set_ylabel("GPU utilisation (%)")
        ax.set_ylim(0, 100)
        ax.set_title(title)
        ax.legend()

        plt.show()

def summarise_resources(data):
    summary = {"mean_cpu_pct": data["cpu"].mean(), "peak_cpu_pct": data["cpu"].max(), "mean_ram_gb": data["ram_gb"].mean(), "peak_ram_gb": data["ram_gb"].max()}

    if data["gpu_util"].notna().any():
        summary["mean_gpu_pct"] = data["gpu_util"].mean()
        summary["peak_gpu_pct"] = data["gpu_util"].max()
    else:
        summary["mean_gpu_pct"] = None
        summary["peak_gpu_pct"] = None

    if data["gpu_gb"].notna().any():
        summary["mean_gpu_gb"] = data["gpu_gb"].mean()
        summary["peak_gpu_gb"] = data["gpu_gb"].max()
    else:
        summary["mean_gpu_gb"] = None
        summary["peak_gpu_gb"] = None

    return summary

### 1.3. Local imports 

In [17]:
from fact_verification.data import CandidateStore, load_claims
from fact_verification.retrieval.bm25 import BM25Config, BM25Retriever

# left unused for practical reasons, DPR is too difficult to run on my hardware in the ideal setup and therefore has been dropped for a simpler dense retrieval model
# from fact_verification.retrieval.dpr import DPRConfig, DPRRetriever

from fact_verification.retrieval.dense import DenseConfig,DenseRetriever
from fact_verification.retrieval.hybrid import HybridConfig, HybridRetriever
from fact_verification.reranking.cross_encoder import CrossEncoderConfig, CrossEncoderReranker
from fact_verification.evaluation import evaluate_retrieval, evaluate_retrieval_per_claim

## 2. Experimental Configuration and Reproducibility

All variables capable of changing the experimental outcome are defined here.
Values should not be changed inside individual retrieval sections.

### 2.1. Main Experiment Configuration

In [18]:
EXPERIMENT = {
    "experiment_name": "initial_retriever_comparison",
    "seed": 67,

    "dataset": {
        "name": "AVeriTeC",
        "split": "dev",
        "revision": None,
        "root": os.environ.get("AVERITEC_ROOT"),
    },

    "candidates": {
        "retrieval_unit": "sentence",
        "chunk_size": None,
        "chunk_overlap": None,
    },

    "retrieval": {
        "retrieve_k": 100,
    },

    "evaluation": {
        "cutoffs": [1, 5, 10, 20, 50, 100],
    },

    "bm25": {
        "k1": 1.5,
        "b": 0.75,
        "epsilon": 0.25,
        "lowercase": False,
    },

    # left unused for practical reasons, DPR is too difficult to run on my hardware in the ideal setup and therefore has been dropped for a simpler dense retrieval model
    # "dpr": {
    #     "question_model":
    #         "facebook/dpr-question_encoder-multiset-base",
    #     "context_model":
    #         "facebook/dpr-ctx_encoder-multiset-base",

    #     "question_revision": None,
    #     "context_revision": None,

    #     "batch_size": 32,
    #     "max_length": 512,
    #     "use_title": False,
    #     "device": "cuda" if torch.cuda.is_available() else "cpu",
    # },

    "dense": {
        "model_name": ("sentence-transformers/multi-qa-MiniLM-L6-cos-v1"),
        "revision": None,
        "batch_size": 64,
        "device": (
            "cuda"
            if torch.cuda.is_available()
            else "cpu"
        ),
    },

    "hybrid": {
        "method": "rrf",
        "fusion_depth": 100,
        "rrf_constant": 60,
        "alpha": 0.5,
    },

    "reranker": {
        "model_name":
            "cross-encoder/ms-marco-MiniLM-L6-v2",
        "revision": None,
        "batch_size": 32,
        "candidate_k": 100,
        "output_k": 20,
        "device": "cuda" if torch.cuda.is_available() else "cpu",
    },
    "run": {
        "mode": "full_test",
        "claims": 500,
        "run_id": "main_20260819",
    },
}
DATA_ROOT = EXPERIMENT["dataset"]["root"]

if DATA_ROOT is None:
    raise RuntimeError("AVERITEC_ROOT is not set \n set it to the root directory of the AVeriTeC data")

DATA_ROOT = Path(DATA_ROOT)



### 2.2. Resolve configuration / paths

In [19]:
DATA_ROOT_VALUE = EXPERIMENT["dataset"]["root"]

REPO_ROOT = Path.cwd().resolve().parent
RESULTS_ROOT = REPO_ROOT / "results"

RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

if DATA_ROOT_VALUE is None:
    raise RuntimeError("AVERITEC_ROOT is not set \n set it to the root directory of the AVeriTeC data")

DATA_ROOT = Path(DATA_ROOT_VALUE).expanduser().resolve()

if not DATA_ROOT.exists():
    raise FileNotFoundError(f"AVeriTeC data root does not exist: {DATA_ROOT}")

print(f"Repository root: {REPO_ROOT}")
print(f"Data root: {DATA_ROOT}")
print(f"Results root: {RESULTS_ROOT}")

Repository root: C:\Users\conno_i4oarur.MACPEPINO\OneDrive\Documents\GitHub\AutomatedFactVerification_RetrieverComparison
Data root: C:\Users\conno_i4oarur.MACPEPINO\OneDrive\Documents\GitHub\AutomatedFactVerification_RetrieverComparison\.local_data\averitec\2ca9dee23a2a
Results root: C:\Users\conno_i4oarur.MACPEPINO\OneDrive\Documents\GitHub\AutomatedFactVerification_RetrieverComparison\results


### 2.3. Fix random seeds

In [20]:
SEED = EXPERIMENT["seed"]

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

### 2.4. Capture environment 

In [21]:
def get_git_commit():
    try:
        return subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
    except Exception:
        return None


ENVIRONMENT = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "python_version": sys.version,
    "torch_version": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "cuda_version": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "git_commit": get_git_commit(),
    "transformers_version": transformers.__version__,
    "sentence_transformers_version":sentence_transformers.__version__,
    "nltk_version": nltk.__version__,
}
pd.Series(ENVIRONMENT)


timestamp_utc                                     2026-08-20T11:22:38.654262+00:00
python_version                   3.14.4 (tags/v3.14.4:23116f9, Apr  7 2026, 14:...
torch_version                                                         2.13.0+cu132
cuda_available                                                                True
cuda_version                                                                  13.2
gpu                                                        NVIDIA GeForce RTX 3060
git_commit                                bd7a6f491c0f0735d5b318a1389f276045375a51
transformers_version                                                        5.15.0
sentence_transformers_version                                                5.7.0
nltk_version                                                                3.10.2
dtype: object

### 2.5. Initialise Run Directory

In [22]:
RUN_DIR = RESULTS_ROOT / EXPERIMENT["run"]["run_id"]
RUN_DIR.mkdir(parents=True, exist_ok=True)

print("Run directory:", RUN_DIR)

Run directory: C:\Users\conno_i4oarur.MACPEPINO\OneDrive\Documents\GitHub\AutomatedFactVerification_RetrieverComparison\results\main_20260819


## 3. Load experimental data
Dataset structure and annotation characteristics are explored separately in
`01_averitec_data_inspection.ipynb`.  (NOTE: or whatever its called later)

Only the processed inputs required for retrieval are loaded here.

### 3.1. Initialise Claims and Candidate Store

In [23]:
claims = load_claims(root=DATA_ROOT, split=EXPERIMENT["dataset"]["split"])


if EXPERIMENT["run"]["mode"] == "pilot":
    claims = claims.sample(n=EXPERIMENT["run"]["claims"], random_state=EXPERIMENT["seed"]).sort_values("claim_id").reset_index(drop=True)

candidate_store = CandidateStore(root=DATA_ROOT, split=EXPERIMENT["dataset"]["split"], retrieval_unit=EXPERIMENT["candidates"]["retrieval_unit"], chunk_size=EXPERIMENT["candidates"]["chunk_size"], chunk_overlap=EXPERIMENT["candidates"]["chunk_overlap"])

if EXPERIMENT["run"]["mode"] == "full_test":
    assert len(claims) == EXPERIMENT["run"]["claims"], (f"expected {EXPERIMENT['run']['claims']} claims, but loaded {len(claims)}")
    assert len(candidate_store) == len(claims), (f"claims: {len(claims)}, knowledge stores: {len(candidate_store)}")

print(f"Claims loaded: {len(claims):,}")
print(f"Knowledge-store claims: {len(candidate_store):,}")
print(f"Retrieval unit: {candidate_store.retrieval_unit}")
print(f"Knowledge store: {candidate_store.directory}")


Claims loaded: 500
Knowledge-store claims: 500
Retrieval unit: sentence
Knowledge store: C:\Users\conno_i4oarur.MACPEPINO\OneDrive\Documents\GitHub\AutomatedFactVerification_RetrieverComparison\.local_data\averitec\2ca9dee23a2a\knowledge_store\dev\output_dev


### 3.2. Candidate Generation Sanity Check

In [24]:
EXAMPLE_CLAIM_ID = claims.iloc[0]["claim_id"]

example_claim = claims.loc[claims["claim_id"] == EXAMPLE_CLAIM_ID, "claim"].iloc[0]
example_candidates = candidate_store.load_claim(EXAMPLE_CLAIM_ID)

print(f"Claim ID: {EXAMPLE_CLAIM_ID}")
print(f"Claim: {example_claim}")
print(f"Candidates generated: {len(example_candidates):,}")

display(example_candidates[["candidate_id", "source_url", "text", "sentence_start", "sentence_end"]].head(10))

Claim ID: 0
Claim: In a letter to Steve Jobs, Sean Connery refused to appear in an apple commercial.
Candidates generated: 534,488


,candidate_id,source_url,text,sentence_start,sentence_end
0,0:0:0,https://web.archive.org/web/20201129141238/htt...,"San Francisco, CA — It’s easy to appreciate iM...",0,0
1,0:0:1,https://web.archive.org/web/20201129141238/htt...,"In fact, Apple’s concerns over the brand-new c...",1,1
2,0:0:2,https://web.archive.org/web/20201129141238/htt...,The surprising story comes to light in iMacula...,2,2
3,0:0:3,https://web.archive.org/web/20201129141238/htt...,"As Ms. Woods spins the tale, it was just weeks...",3,3
4,0:0:4,https://web.archive.org/web/20201129141238/htt...,"Steve Jobs, a lifelong fan of James Bond (he’d...",4,4
5,0:0:5,https://web.archive.org/web/20201129141238/htt...,"“The ad was of dubious quality, clearly not on...",5,5
6,0:0:6,https://web.archive.org/web/20201129141238/htt...,"Though Steve had a thing for Sean Connery, the...",6,6
7,0:0:7,https://web.archive.org/web/20201129141238/htt...,Connery’s final rejection was accompanied by a...,7,7
8,0:0:8,https://web.archive.org/web/20201129141238/htt...,"Needless to say, iMac managed to “survive” wit...",8,8
9,0:0:9,https://web.archive.org/web/20201129141238/htt...,iMaculate Conception reveals a number of other...,9,9


#### 3.2.1. Candidate store sanity checks

In [25]:
assert not example_candidates.empty
assert example_candidates["candidate_id"].is_unique
assert example_candidates["text"].notna().all()

required_columns = {"claim_id", "candidate_id", "source_url", "text"}
missing = required_columns - set(example_candidates.columns)

assert not missing, (f"Candidate store is missing required fields: {sorted(missing)}")

### 3.4. Initialise Run State

In [26]:
RUNTIMES = {}
RESOURCE_USAGE = {}
RUN_SUMMARY = {}

def validate_ranked_results(results, expected_claims, expected_k, method):
    assert results["claim_id"].nunique() == expected_claims, (f"{method}: expected {expected_claims} claims, found {results['claim_id'].nunique()}")

    counts = results.groupby("claim_id").size()

    assert (counts == expected_k).all(), (f"{method}: not every claim contains {expected_k} ranked results")
    assert results["rank"].between(1, expected_k).all()

    print(f"{method} output validated.")

## 4. Retrieval Experiments
Each retrieval method is executed using the same claims, candidate store and
retrieval depth defined in the experimental configuration.

### 4.1. BM25 Retriever

#### 4.1.1. Initialise BM25 Retriever

In [27]:
bm25 = BM25Retriever(config=BM25Config(**EXPERIMENT["bm25"]))

print("BM25 retriever initialised.")

BM25 retriever initialised.


#### 4.1.1. Run BM25 

Also initialises runtime dictionaries (RUNTIMES, RESOURCE_USAGE, RUN_SUMMARY)

In [28]:
BM25_RESULTS_PATH = RUN_DIR / "bm25_results.pkl"
BM25_RESOURCE_PATH = RUN_DIR / "bm25_resource_usage.pkl"
BM25_STATE_PATH = RUN_DIR / "bm25_state.pkl"


if (BM25_RESULTS_PATH.exists() and BM25_RESOURCE_PATH.exists() and BM25_STATE_PATH.exists()):
    print("BM25 already completed. Loading saved results...")

    bm25_results = pd.read_pickle(BM25_RESULTS_PATH)
    RESOURCE_USAGE["bm25"] = pd.read_pickle(BM25_RESOURCE_PATH)

    bm25_state = pd.read_pickle(BM25_STATE_PATH)

    RUNTIMES["bm25"] = bm25_state["runtime_seconds"]
    RUN_SUMMARY["bm25"] = bm25_state["summary"]

else:
    print("Running BM25 retrieval...")

    monitor = ResourceMonitor()
    monitor.start()

    start = perf_counter()

    bm25_results = bm25.retrieve_batch(claims=claims, candidates=candidate_store, k=EXPERIMENT["retrieval"]["retrieve_k"])

    RUNTIMES["bm25"] = perf_counter() - start
    RESOURCE_USAGE["bm25"] = monitor.stop()

    RUN_SUMMARY["bm25"] = {"claims": len(claims), "results": len(bm25_results), "runtime_seconds": RUNTIMES["bm25"], **summarise_resources(RESOURCE_USAGE["bm25"])}

    # Persist immediately after successful completion.
    bm25_results.to_pickle(BM25_RESULTS_PATH)

    RESOURCE_USAGE["bm25"].to_pickle(BM25_RESOURCE_PATH)

    pd.to_pickle({"runtime_seconds": RUNTIMES["bm25"], "summary": RUN_SUMMARY["bm25"]}, BM25_STATE_PATH)

    print("BM25 retrieval completed and saved.")


print(f"BM25 results: {len(bm25_results):,}")
print(f"Runtime: {RUNTIMES['bm25']:.2f}s")

plot_resources(RESOURCE_USAGE["bm25"], "BM25 Resource Usage")

validate_ranked_results(bm25_results, len(claims), 100, "BM25")

Running BM25 retrieval...


BM25Retriever:   3%|▎         | 14/500 [08:39<5:00:17, 37.07s/claim]


KeyboardInterrupt: 

### 4.2. Dense Retriever

#### 4.2.1. Initialise Dense Retriever

In [ ]:
dense = DenseRetriever(config=DenseConfig(**EXPERIMENT["dense"]))

print("Dense retriever initialised.")
print("Device:", dense.model.device)

#### 4.2.2. Run Dense Retriever

In [ ]:
DENSE_RESULTS_PATH = RUN_DIR / "dense_results.pkl"
DENSE_CHECKPOINT_PATH = RUN_DIR / "dense_checkpoint.pkl"
DENSE_RESOURCE_PATH = RUN_DIR / "dense_resource_usage.pkl"
DENSE_STATE_PATH = RUN_DIR / "dense_state.pkl"

CHECKPOINT_EVERY = 10

# load a completed dense run if one already exists
if (DENSE_RESULTS_PATH.exists() and DENSE_RESOURCE_PATH.exists() and DENSE_STATE_PATH.exists()):
    print("Dense retrieval already completed. Loading saved results...")
    
    dense_results = pd.read_pickle(DENSE_RESULTS_PATH)
    RESOURCE_USAGE["dense"] = pd.read_pickle(DENSE_RESOURCE_PATH)
    dense_state = pd.read_pickle(DENSE_STATE_PATH)

    RUNTIMES["dense"] = dense_state["runtime_seconds"]
    RUN_SUMMARY["dense"] = dense_state["summary"]

# otherwise start or resume an incomplete dense run
else:
    # load previous retrieval progress
    if DENSE_CHECKPOINT_PATH.exists():
        dense_results = pd.read_pickle(DENSE_CHECKPOINT_PATH)
        completed_claims = set(dense_results["claim_id"].unique())

        print(f"Resuming dense retrieval: {len(completed_claims):,} claims already complete")

    else:
        dense_results = pd.DataFrame()
        completed_claims = set()

        print("Starting dense retrieval from the beginning.")

    # load previous runtime state
    if DENSE_STATE_PATH.exists():
        dense_state = pd.read_pickle(DENSE_STATE_PATH)
        previous_runtime = dense_state.get("runtime_seconds", 0.0)

    else:
        previous_runtime = 0.0

    # load previous resource samples
    if DENSE_RESOURCE_PATH.exists():
        previous_usage = pd.read_pickle(DENSE_RESOURCE_PATH)
    else:
        previous_usage = pd.DataFrame()

    # determine which claims still require retrieval
    remaining_claims = claims[~claims["claim_id"].isin(completed_claims)]

    print(f"Remaining claims: {len(remaining_claims):,}")

    # Start resource monitoring
    monitor = ResourceMonitor()
    monitor.start()

    start = perf_counter()
    try:
        # run remaining claims
        for claim_index, (_, claim) in enumerate(tqdm(remaining_claims.iterrows(), total=len(remaining_claims), desc="DenseRetriever", unit="claim"), start=1):
            claim_id = claim["claim_id"]
            claim_candidates = candidate_store.load_claim(claim_id)
            claim_candidates = claim_candidates.drop(columns=["claim_id"], errors="ignore")

            result = dense.retrieve(query=claim["claim"], candidates=claim_candidates, k=EXPERIMENT["retrieval"]["retrieve_k"])
            result.insert(0, "claim_id", claim_id)
            dense_results = pd.concat([dense_results, result], ignore_index=True)

            # checkpoint every N completed claims

            if (claim_index % CHECKPOINT_EVERY == 0 or claim_index == len(remaining_claims)):
                dense_results.to_pickle(DENSE_CHECKPOINT_PATH)

                current_runtime = (previous_runtime + (perf_counter() - start))

                # snapshot resource samples collected during the current session
                current_usage = pd.DataFrame(monitor.samples).copy()

                if not current_usage.empty:
                    current_usage["time"] += (previous_runtime)

                checkpoint_usage = pd.concat([previous_usage, current_usage], ignore_index=True)
                checkpoint_usage.to_pickle(DENSE_RESOURCE_PATH)


                pd.to_pickle({"runtime_seconds": current_runtime, "completed_claims": int(dense_results["claim_id"].nunique())}, DENSE_STATE_PATH)

    # if retrieval fails, preserve all completed work before reraising exception
    except Exception:
        session_usage = monitor.stop()
        current_runtime = (previous_runtime + (perf_counter() - start))
        dense_results.to_pickle(DENSE_CHECKPOINT_PATH)

        if not session_usage.empty:
            session_usage = session_usage.copy()

            session_usage["time"] += (previous_runtime)

        checkpoint_usage = pd.concat([previous_usage, session_usage], ignore_index=True)
        checkpoint_usage.to_pickle(DENSE_RESOURCE_PATH)


        pd.to_pickle({"runtime_seconds": current_runtime, "completed_claims": int(dense_results["claim_id"].nunique())}, DENSE_STATE_PATH)

        print("dense retrieval interrupted, completed progress has been checkpointed")
        raise

    # finalise successful dense retrieval
    session_usage = monitor.stop()
    session_runtime = (perf_counter() - start)
    RUNTIMES["dense"] = (previous_runtime + session_runtime)


    if not session_usage.empty:
        session_usage = session_usage.copy()
        session_usage["time"] += (previous_runtime)


    RESOURCE_USAGE["dense"] = pd.concat([previous_usage, session_usage], ignore_index=True)
    dense_results = (dense_results.sort_values(["claim_id", "rank"]).reset_index(drop=True))
    RUN_SUMMARY["dense"] = {"claims": len(claims), "results": len(dense_results), "runtime_seconds": RUNTIMES["dense"], **summarise_resources(RESOURCE_USAGE["dense"])}

    # save completed dense run
    dense_results.to_pickle(DENSE_RESULTS_PATH)
    RESOURCE_USAGE["dense"].to_pickle(DENSE_RESOURCE_PATH)

    pd.to_pickle({"runtime_seconds": RUNTIMES["dense"], "summary": RUN_SUMMARY["dense"], "completed_claims": int(dense_results["claim_id"].nunique())}, DENSE_STATE_PATH)

    # final result now exists, so partial checkpoint no longer required
    if DENSE_CHECKPOINT_PATH.exists():
        DENSE_CHECKPOINT_PATH.unlink()

    print("dense retrieval completed and saved")

# validate loaded or newly generated output
expected_claims = len(claims)

expected_k = (EXPERIMENT["retrieval"]["retrieve_k"])
actual_claims = (dense_results["claim_id"].nunique())
results_per_claim = (dense_results.groupby("claim_id").size())

assert actual_claims == expected_claims, (f"dense: expected {expected_claims} claims, found {actual_claims}")
assert (results_per_claim == expected_k).all(), (f"dense: not every claim contains {expected_k} retrieved candidates")
assert dense_results["rank"].between(1, expected_k).all(), ("dense: invalid rank values detected")

# display summary
print("Dense retrieval output validated.")
print(f"Dense results: {len(dense_results):,}")
print(f"Dense claims: {actual_claims:,}")
print(f"Runtime: {RUNTIMES['dense']:.2f}s ({RUNTIMES['dense'] / 3600:.2f}h)")

plot_resources(RESOURCE_USAGE["dense"], "Dense Retrieval Resource Usage")

### 4.3. Hybrid Retriever

#### 4.3.1. Initialise Hybrid Retriever
This section essentially just consumes the existing results, opposed to regenerating the experiment results for both BM25 and DPR

In [ ]:
hybrid = HybridRetriever(lexical_retriever=bm25, dense_retriever=dense, config=HybridConfig(**EXPERIMENT["hybrid"]))

print("Hybrid retriever initialised.")

Hybrid retriever initialised.


#### 4.3.2. Run Hybrid Fusion

In [ ]:
HYBRID_RESULTS_PATH = RUN_DIR / "hybrid_results.pkl"
HYBRID_STATE_PATH = RUN_DIR / "hybrid_state.pkl"


#  load completed hybrid results if available
if (HYBRID_RESULTS_PATH.exists() and HYBRID_STATE_PATH.exists()):
    print("Hybrid fusion already completed. Loading saved results...")

    hybrid_results = pd.read_pickle(HYBRID_RESULTS_PATH)
    hybrid_state = pd.read_pickle(HYBRID_STATE_PATH)

    RUNTIMES["hybrid_fusion"] = hybrid_state["fusion_runtime_seconds"]
    RUNTIMES["hybrid_total"] = hybrid_state["total_runtime_seconds"]
    RUN_SUMMARY["hybrid"] = hybrid_state["summary"]


# otherwise run the fusion
else:
    print("Running hybrid RRF fusion...")
    start = perf_counter()

    hybrid_results = hybrid.fuse_batch(lexical_results=bm25_results, dense_results=dense_results, k=EXPERIMENT["retrieval"]["retrieve_k"])

    RUNTIMES["hybrid_fusion"] = perf_counter() - start
    RUNTIMES["hybrid_total"] = RUNTIMES["bm25"] + RUNTIMES["dense"] + RUNTIMES["hybrid_fusion"]
    RUN_SUMMARY["hybrid"] = {"claims": len(claims), "results": len(hybrid_results), "runtime_seconds": RUNTIMES["hybrid_fusion"], "total_runtime_seconds": RUNTIMES["hybrid_total"]}


    # save completed hybrid output
    hybrid_results.to_pickle(HYBRID_RESULTS_PATH)

    pd.to_pickle({"fusion_runtime_seconds": RUNTIMES["hybrid_fusion"], "total_runtime_seconds": RUNTIMES["hybrid_total"], "summary": RUN_SUMMARY["hybrid"]}, HYBRID_STATE_PATH)
    print("Hybrid fusion completed and saved.")

# display results
print(f"Hybrid results: {len(hybrid_results):,}")
print(f"Fusion runtime: {RUNTIMES['hybrid_fusion']:.4f}s")
print(f"Total hybrid pipeline runtime: {RUNTIMES['hybrid_total'] / 3600:.2f}h")

validate_ranked_results(hybrid_results, len(claims), 100, "Hybrid")

# release first stage retrieval models before loading crossencoder
if "hybrid" in globals():
    del hybrid

if "dense" in globals():
    del dense

if "bm25" in globals():
    del bm25

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()


### 4.4. Cross-Encoder Reranker

#### 4.4.1. Initialse Cross-Encoder Reranker
As with the hybrid retrieval, the reranker should consume candidate results from one of the first-stage systems

In [ ]:
reranker_config = EXPERIMENT["reranker"]
cross_encoder = CrossEncoderReranker(config=CrossEncoderConfig(model_name=reranker_config["model_name"], revision=reranker_config["revision"], batch_size=reranker_config["batch_size"], device=reranker_config["device"]))
print("Cross-encoder reranker initialised.")

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 5304.77it/s]


Cross-encoder reranker initialised.


#### 4.4.2. Run Cross-Encoder Reranking

In [ ]:
CROSS_ENCODER_RESULTS_PATH = RUN_DIR / "cross_encoder_results.pkl"
CROSS_ENCODER_RESOURCE_PATH = RUN_DIR / "cross_encoder_resource_usage.pkl"
CROSS_ENCODER_STATE_PATH = RUN_DIR / "cross_encoder_state.pkl"


# load completed results if available

if (CROSS_ENCODER_RESULTS_PATH.exists() and CROSS_ENCODER_RESOURCE_PATH.exists() and CROSS_ENCODER_STATE_PATH.exists()):
    print("Cross-encoder reranking already completed. Loading saved results...")

    reranked_results = pd.read_pickle(CROSS_ENCODER_RESULTS_PATH)

    RESOURCE_USAGE["cross_encoder"] = pd.read_pickle(CROSS_ENCODER_RESOURCE_PATH)

    cross_encoder_state = pd.read_pickle(CROSS_ENCODER_STATE_PATH)

    RUNTIMES["cross_encoder"] = cross_encoder_state["runtime_seconds"]
    RUNTIMES["cross_encoder_total"] = cross_encoder_state["total_runtime_seconds"]
    RUN_SUMMARY["cross_encoder"] = cross_encoder_state["summary"]


# otherwise run reranker

else:
    print("Running cross-encoder reranking...")

    monitor = ResourceMonitor()
    monitor.start()

    start = perf_counter()

    reranked_results = cross_encoder.rerank_batch(claims=claims, candidates=hybrid_results,candidate_k=EXPERIMENT["reranker"]["candidate_k"], output_k=EXPERIMENT["reranker"]["output_k"])

    RUNTIMES["cross_encoder"] = perf_counter() - start

    RESOURCE_USAGE["cross_encoder"] = monitor.stop() 


    # total cost of producing final reranked ranking
    RUNTIMES["cross_encoder_total"] = RUNTIMES["hybrid_total"] + RUNTIMES["cross_encoder"]

    RUN_SUMMARY["cross_encoder"] = {"claims": len(claims), "results": len(reranked_results), "runtime_seconds": RUNTIMES["cross_encoder"], "total_runtime_seconds": RUNTIMES["cross_encoder_total"], **summarise_resources(RESOURCE_USAGE["cross_encoder"])}


    # persist completed outputs immediately
    reranked_results.to_pickle(CROSS_ENCODER_RESULTS_PATH)

    RESOURCE_USAGE["cross_encoder"].to_pickle(CROSS_ENCODER_RESOURCE_PATH)

    pd.to_pickle({"runtime_seconds": RUNTIMES["cross_encoder"], "total_runtime_seconds": RUNTIMES["cross_encoder_total"], "summary": RUN_SUMMARY["cross_encoder"]}, CROSS_ENCODER_STATE_PATH)
    print("Cross-encoder reranking completed and saved.")

# display results

print(f"Reranked results: {len(reranked_results):,}")
print(f"Cross-encoder runtime: {RUNTIMES['cross_encoder']:.2f}s")
print(f"Complete reranked pipeline runtime: {RUNTIMES['cross_encoder_total'] / 3600:.2f}h")
display(reranked_results.head())

plot_resources(RESOURCE_USAGE["cross_encoder"], "Cross-Encoder Resource Usage")

validate_ranked_results(reranked_results, len(claims), 20, "Cross-Encoder")

### 4.5. Runtime Summary

In [ ]:
run_summary = pd.DataFrame(RUN_SUMMARY).T.reset_index(names="method")
display(run_summary)


scale_factor = (500 / len(claims))

run_summary["estimated_full_runtime_minutes"] = run_summary["runtime_seconds"] * scale_factor / 60
run_summary["estimated_full_runtime_hours"] = run_summary["estimated_full_runtime_minutes"] / 60

display(run_summary)

## 6. Comparative Retrieval Evaluation


###  6.1. Aggregate Retrieval Evaluation

In [ ]:
EVALUATION_SUMMARIES = {}

cutoffs = EXPERIMENT["evaluation"]["cutoffs"]
EVALUATION_SUMMARIES["bm25"] = evaluate_retrieval(claims=claims, results=bm25_results, cutoffs=cutoffs)
EVALUATION_SUMMARIES["dense"] = evaluate_retrieval(claims=claims, results=dense_results, cutoffs=cutoffs)
EVALUATION_SUMMARIES["hybrid"] = evaluate_retrieval(claims=claims, results=hybrid_results, cutoffs=cutoffs)

reranker_cutoffs = [k for k in cutoffs if k <= EXPERIMENT["reranker"]["output_k"]]

EVALUATION_SUMMARIES["cross_encoder"] = evaluate_retrieval(claims=claims, results=reranked_results, cutoffs=reranker_cutoffs)

### 6.2. Per-Claim Evaluation

In [ ]:
PER_CLAIM_METRICS = {}

PER_CLAIM_METRICS["bm25"] = evaluate_retrieval_per_claim(claims, bm25_results, cutoffs)
PER_CLAIM_METRICS["dense"] = evaluate_retrieval_per_claim(claims, dense_results, cutoffs)
PER_CLAIM_METRICS["hybrid"] = evaluate_retrieval_per_claim(claims, hybrid_results, cutoffs)
PER_CLAIM_METRICS["cross_encoder"] = evaluate_retrieval_per_claim(claims, reranked_results, reranker_cutoffs)

### 6.3. Evaluation Summay

In [ ]:
evaluation_table = pd.DataFrame(EVALUATION_SUMMARIES).T.reset_index(names="method")
display(evaluation_table)

### 6.4. Evaluation Sanity Checks

In [ ]:
assert all(summary["NumClaims"] == len(claims) for summary in EVALUATION_SUMMARIES.values())

evaluable_counts = {method: int(summary["NumEvaluableClaims"]) for method, summary in EVALUATION_SUMMARIES.items()}
print("Evaluable claims:", evaluable_counts)

assert len(set(evaluable_counts.values())) == 1

display(PER_CLAIM_METRICS["bm25"].head())

## 7. Save Experimental Run

### 7.1. Save Ranked Retrieval Results

In [ ]:
RESULT_SETS = {"bm25": bm25_results, "dense": dense_results, "hybrid": hybrid_results, "cross_encoder": reranked_results}

for method, results in RESULT_SETS.items():
    results.to_pickle(RUN_DIR / f"{method}_results.pkl")

### 7.2. Save Evaluation Results

In [ ]:
evaluation_table.to_csv(RUN_DIR / "evaluation_summary.csv", index=False)

for method, metrics in PER_CLAIM_METRICS.items():
    metrics.to_pickle(RUN_DIR / f"{method}_per_claim_metrics.pkl")

### 7.3. Save Resource Measurements

In [ ]:
for method, usage in RESOURCE_USAGE.items():
    if isinstance(usage, pd.DataFrame):
        usage.to_pickle(RUN_DIR / f"{method}_resource_usage.pkl")

pd.DataFrame(RUN_SUMMARY).T.to_csv(RUN_DIR / "resource_summary.csv")

### 7.4. Save Experiment Configuration

In [ ]:
def json_serialisable(value):
    if isinstance(value, Path):
        return str(value)

    if isinstance(value, dict):
        return {key: json_serialisable(item) for key, item in value.items()}

    if isinstance(value, (list, tuple)):
        return [json_serialisable(item) for item in value]

    return value

with open(RUN_DIR / "experiment_config.json", "w", encoding="utf-8") as f:
    json.dump(json_serialisable(EXPERIMENT), f, indent=2)

with open(RUN_DIR / "environment.json", "w", encoding="utf-8") as f:
    json.dump(json_serialisable(ENVIRONMENT), f, indent=2)

### 7.5. Verify Saved Run

In [ ]:
print(f"Saved experiment to: {RUN_DIR}")

for path in sorted(RUN_DIR.iterdir()):
    print(f"  {path.name}")